In [1]:
# Importing necessary libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import math
from scipy.spatial.transform import Rotation as R
from scipy.linalg import block_diag
import timeit as timee
import import_ipynb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
import matplotlib.collections
#import effort_solver_nn_3_triangles_3D_stiffness_information_Classical_FEM_changed as solv
import Functions_for_material_law as ML
from sklearn.preprocessing  import MinMaxScaler
from datetime import datetime
start_time = datetime.now()
from mpl_toolkits.mplot3d import Axes3D
import sympy as sym
from matplotlib.patches import Rectangle
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
#import plotly.graph_objects as go
import pickle
from tensorflow.keras.models import Model, load_model
import keras_lmu
%matplotlib qt

importing Jupyter notebook from Functions_for_material_law.ipynb


In [2]:
def props_update(mat_params, i,l,j,k, pois):
    
    return (mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]*pois)/((1 + pois)*(1 - 2*pois)), 0.5*mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 + pois)


def tens_to_voigt(tens):
    
    stress    = np.zeros(shape=(6,))
    
    stress[0] = tens[0][0]
    stress[1] = tens[1][1]
    stress[2] = tens[2][2]
    stress[3] = tens[1][2]
    stress[4] = tens[0][2]
    stress[5] = tens[0][1]
    
    return stress


def voigt_to_tens(stress):
    
    tens = np.zeros(shape=(3,3))
    
    
    tens[0][0] = stress[0]
    tens[1][1] = stress[1]
    tens[2][2] = stress[2]
    tens[0][1] = stress[5]
    tens[0][2] = stress[4]
    tens[1][0] = stress[5]
    tens[1][2] = stress[3]
    tens[2][0] = stress[4]
    tens[2][1] = stress[3]
    
    
    return tens


def split(tens):
   
    I      = np.eye(3,3)
    
    aa     = voigt_to_tens(tens)
    
    shydro = (1/3*(aa.trace()))
    
    return aa - shydro*I, shydro*I


In [3]:
# Initializing all the material properties

mat_params    = {}
I             = np.eye(3,3)
pois          = 0.3
timeincrement = 1e-8
N             = 8.15
m             = 0
n             = 0
for i in range(1):
    
    for l in range(1):
    
        for j in range(1):

            for k in range(1):


                mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0 
                mat_params['stress_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['flow'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = np.zeros(shape=(3,3))
                mat_params['eps_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(3,3))
                mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3))
                mat_params['stress_n'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(6,))
                mat_params['stress_yield'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 180
                mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = 113066
                mat_params['delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                         = np.zeros(shape=(6,))
                mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = np.zeros(shape=(6,))
                #mat_params['strain_n_1']                                                   = np.zeros(shape=(6,))
                mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]   = props_update(mat_params, i,l,j,k, pois) 
                mat_params['pois'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = pois
                mat_params['stress_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = np.zeros(shape=(6,))
                mat_params['eps_vec'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                           = np.zeros(shape=(6,))
                aaa, bbb = mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]
                aa = 2*bbb + aaa
                mat_params['stress_K'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = 11.45
                mat_params['abs_stress'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['stress_back'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.zeros(shape=(3,3))
                mat_params['abs_strain'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])
                mat_params['C_evp_fourth'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3,3,3))
                mat_params['C_evp'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)].copy()
                mat_params['abs_eqpl'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                   = []                                              
                mat_params['C_ep'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                                    = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])
                mat_params['eps_dot'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                    = 0
                mat_params['X_dot'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = np.zeros(shape=(3,3))
                mat_params['D_dot'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = 0
                mat_params['EQPLAS'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.sqrt((np.tensordot(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)],mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]))*(2/3))
                mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]               = np.zeros(shape=(3,3)) 
                K                                                                          = mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 - 2*pois)
                mat_params['Delta_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0.0
                mat_params['R'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0.0
                mat_params['b1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.
                mat_params['b2'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.
                mat_params['D'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0
                mat_params['D_cr'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = 0.22
                mat_params['e_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 0.0
                mat_params['e_r'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 0.16296296296296295
                mat_params['a'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 98939.30
                mat_params['s'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 1533.41
            #mat_params['stiff_point'+str(i+1)+str(l+1)+str(j+1)]                           = np.zeros(shape=(DOF_per_node*nodes_per_element, DOF_per_node*nodes_per_element))
    
    
    #ax['element'+str(i+1)]                                                    = np.zeros(shape=(DOF_per_node*nodes_per_element ,1))
    
#     nn['element_force'+str(i+1)]                                              = []
#     nn['element_disp'+str(i+1)]                                               = []
#     nn['element_stiff'+str(i+1)]                                              = []

In [4]:
units         = 16

state_h  =  tf.zeros(shape=(1,units))
state_c  =  tf.zeros(shape=(1,100))

state_h1 =  tf.zeros(shape=(1,units))
state_c1 =  tf.zeros(shape=(1,100))

state_h2 =  tf.zeros(shape=(1,units))
state_c2 =  tf.zeros(shape=(1,100))

state_h3 =  tf.zeros(shape=(1,64))
state_c3 =  tf.zeros(shape=(1,64))

state_inputs  = [state_h, state_c, state_h1, state_c1]

In [5]:
cell=keras_lmu.LMUCell(memory_d=10,
    order=10,
    theta=784,
    hidden_cell=tf.keras.layers.SimpleRNNCell(units=16, activation='tanh'))

LMU= tf.keras.layers.RNN(cell, return_sequences=True, return_state=True)

cell1=keras_lmu.LMUCell(memory_d=10,
    order=10,
    theta=784,
    hidden_cell=tf.keras.layers.SimpleRNNCell(units=16, activation='tanh'))

LMU1= tf.keras.layers.RNN(cell1, return_sequences=True, return_state=True)

cell2=keras_lmu.LMUCell(memory_d=10,
    order=10,
    theta=784,
    hidden_cell=tf.keras.layers.SimpleRNNCell(units=48, activation='tanh'))

LMU2 = tf.keras.layers.RNN(cell2, return_sequences=True, return_state=True)


cell2=keras_lmu.LMUCell(memory_d=64,
    order=1,
    theta=784,
    hidden_cell=tf.keras.layers.SimpleRNNCell(units=64),)

d1                   = Dense(units = 16, activation='tanh')
d2                   = Dense(units = 16, activation='tanh')

decoder = tf.keras.layers.RNN(cell2, return_sequences=True, return_state=True)

X_input               = Input(shape=(None,2,))
model_state_input_h   = Input(shape=(units,))
model_state_input_c   = Input(shape=(100,))
model_state_input_h1  = Input(shape=(units,))
model_state_input_c1  = Input(shape=(100,))
model_state_input_h2  = Input(shape=(units,))
model_state_input_c2  = Input(shape=(100,))
model_state_input_h3  = Input(shape=(64,))
model_state_input_c3  = Input(shape=(64,))

state_inputs1         = [model_state_input_h, model_state_input_c, model_state_input_h1, model_state_input_c1, model_state_input_h2, model_state_input_c2]

X1, h, c             = LMU(X_input)

X1, h1, c1           = LMU1(X1)

X1                   = d1(X1)
X1                   = d2(X1)
# X1                   = Dense(units = 16, activation='tanh')(X1)
# X1                   = Dense(units = 16, activation='tanh')(X1)
o                    = Dense(units = 1, activation='tanh')(X1)
#o                    = tf.reduce_mean(X1, axis=2, keepdims=True)

model                = Model(inputs = [X_input, model_state_input_h, model_state_input_c, model_state_input_h1, model_state_input_c1], outputs = [o, h, c, h1, c1])

opt                  = Adam(1e-9)


In [6]:
mat_params['mu1111']

43486.92307692308

In [7]:
# delta_st = []
# for uu in range(12000):
    
#     if uu<2000:
        
#         delta_st.append(np.array([-1.13662218e-04,  1.37489468e-07,  5.34234067e-05,  1.94601544e-06,
#         2.55335799e-06,  6.21437461e-07]))
        
#     elif uu>=2000 and uu<4000:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)
    
#     elif uu>=4000 and uu<6000:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)     
        
        
#     elif uu>6000 and uu<=8000:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)        
#     elif uu>8000 and uu<=10000:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)     
        
        
#     elif uu>10000 and uu<=12000:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)   
        

#     else:
        
#         delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)   
        
        

In [8]:
def explicit_integration(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr):
    
    #Delta_pl_L    =   np.zeros(shape=(2,1,1))
    
    #Delta_pl_U1   =   ((((stress_eqv_trial/(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))

    #Delta_pl_U    =  Delta_pl_U1
    
    Delta_pl_U     =  mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement
    #D_n_plus_1     =  mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['D_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement)
                        
                        
                        
    #stress_back_n_plus_1 = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['X_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement
    #stress_n_plus_1      = (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - tens_to_voigt(mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*timeincrement) + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
                        
    
    return Delta_pl_U, mat_params

In [10]:
delta_st  = np.genfromtxt(r'C:\Users\Tandale\sciebo\Prashant_Work\Data\strain44111.csv', delimiter=',')*10*2

In [11]:
l = 0
m = 0
n = 0
random   = 0
Delta_pl = 0
wahl    = 0
deqpl   = []
deqpl1           = []
deqpl2           = []
eps33_ana        = []
eps33_ana_pl     = []
count_tr         = 0
sequence_tracker = 0
trial_stress = []
back_stress  = []
number_of_iteration = 0
start_time = datetime.now()

increment_s = [500000, 500000, 500000]

sign   =  [1, -1, 1]

for ll in range(len(sign)):
    
    print('......................................................................................................')
    print('Load Sequence '+str(l+1)+' The sign is '+str(sign[l]))
    print('......................................................................................................')
    
    for t in range(increment_s[ll]):


        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                    = delta_st*1*sign[ll]
        
        #print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])

        for l in range((1)):

            for m in range((1)):

                for n in range((1)):

                    mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    sig_tr_i_1                                                        = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
                    #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                    stress_back_trial                                                 = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    stress_dev_trial,stress_sph                                        = split(sig_tr_i_1)
                    #if i==0 and m==0 and n==0:
                    #    print(mat_params['stress_n111'])
                    sph_                                                              = (1/3)*(stress_sph.trace())
                    zeta                                                              = stress_dev_trial-stress_back_trial
                    stress_eqv_trial                                                  = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                    N_dirr                                                            = ((stress_dev_trial - stress_back_trial)/stress_eqv_trial)
                    J_s                                                               = np.sqrt(1.5*(np.tensordot(stress_dev_trial, stress_dev_trial)))    
                    
                    # Write a function to give the current yield stress
                    #mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    #print(mat_params)

                    if stress_eqv_trial/(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  > abs(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]+mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]):
                        
                        #print('Wir befinden uns weiter im Nichtlinearen Bereich')
                        #print('Hello')
                        N_dirr_bef = (N_dirr.copy())*stress_eqv_trial
                        if count_tr==0 and mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0][0]==0:
                            sequence_tracker=0
                            #mat_params['D1111'] = 0.004
                            count_tr=1

                        elif count_tr==0 and mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0][0]!=0:
                            sequence_tracker=1
                            count_tr=1

                        for p in range(1000):
                            
                            

                            ##if not(p==0):

                                #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                                #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #stress_dev_trial                                              = mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
                                #if i==0 and m==0 and n==0:

                                #print('hello')
                                #    print(mat_params['stress_n111'])
                            
                            sigma33_NN                               = sig_tr_i_1.copy()
                            
                            x_batch_train                            = np.array([[mat_params['delta_eps1111'][2], -(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] + mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])]])
                            
                            x_batch_train                            = tf.cast((x_batch_train), dtype = tf.float32)
                            x_batch_train                            = tf.reshape(x_batch_train, shape=(1,1,2))
                            
                            with tf.GradientTape(persistent=True) as tt:
                            
                                F_n, h, c, h1, c1    = model([x_batch_train]+state_inputs)
                                tt.watch(model.trainable_weights)
                            
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]= F_n[0][0][0]
                                sig_tr_i_1                                                    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                stress_dev_trial, stress_sph                                  = split(sig_tr_i_1)
                                stress_eqv_trial                                              = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                                N_dirr                                                        = (stress_dev_trial-stress_back_trial)/stress_eqv_trial
                                sph_                                                          = (1/3)*(stress_sph.trace())
                                
                                #Delta_pl, mat_params, q, sigma_star_N                         = implicit_integration_vp_1(mat_params, i, N, timeincrement, 1, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n,N_dirr)
                                #Delta_pl, mat_params, q, sigma_star_N                         = implicit_integration_vp(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr)

                                Delta_pl, mat_params                                          = explicit_integration(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr)

                                #D_sigma33                                                     = mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['D_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement)
                                Delta_pl_sigma33                                              = mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement

                                loss_value                                                    = tf.math.pow(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] + (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][0]*mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0]*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) +mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][1]*mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1]*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2]*F_n[0][0][0]*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) - (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_sigma33[2][2], 2)
                                
                            grads_batch      = tt.gradient(loss_value, model.trainable_weights)
                            opt.apply_gradients(zip(grads_batch,model.trainable_weights))
                            #print(loss_value)
                            
                                
                                #amp                                                           = np.array([1.5, 1.5, 1.5, 3, 3, 3])
                                #sigma_i_1                                                     = sigma_33_trial - mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2:3,:].dot(amp*tens_to_voigt(Delta_pl*N_dirr))[0]
                                
                            #delta_eps33_i_1                                               = delta_eps33_i_minus_1 -(((sigma_i)*(delta_eps33_i - delta_eps33_i_minus_1))/(1e-9 + sigma_i_1 - sigma_i))


                                
#                                 mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = delta_eps33_i_1
#                                 delta_eps33_i_minus_1 =  delta_eps33_i
#                                 delta_eps33_i         =  delta_eps33_i_1
#                                 sigma_i               =  sigma_i_1
    #                             a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
    #                             a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
    #                             trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
    #                             back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])
    #                             if sequence_tracker==0:
    #                                 deqpl1.append(Delta_pl)

    #                             elif sequence_tracker==1:

    #                                 deqpl2.append(Delta_pl)
                                #print(abs(delta_eps33_i - delta_eps33_i_minus_1)/abs(delta_eps33_i_1))

                            if abs(loss_value) < 1e-9:

                                #print('Wir befinden uns weiter im Nichtlinearen Bereich und Konvergieren in '+str(p+1)+' Iterations', 'zeitinkrement'+str(t+1))   
                                #deqpl1.append(Delta_pl)
                                #a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
                                #a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
                                #trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
                                #back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = F_n[0][0][0].numpy()
                                state_inputs = [h, c, h1, c1]
                                #eps33_ana.append(delta_eps33_i_1)
                                break

                                #print('Trial_Stress '+str(sigma_33_trial),'Sigma_i_1 ' +str(sigma_i_1), 'Equivalent Plastic Strain: '+str(Delta_pl))
                                



                            #elif p==0:

    #                             a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
    #                             a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
    #                             trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
    #                             back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])





                                #sigma_33_trial        = sig_tr_i_1[2]
                                #mat_params, Delta_pl  = implicit_integration(mat_params, stress_eqv_trial,i,l,m,n)
                                #Delta_pl, mat_params, q, sigma_star_N  = implicit_integration_vp_1(mat_params, i, N, timeincrement, 1, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n,N_dirr)
                                #Delta_pl, mat_params, q, sigma_star_N  = implicit_integration_vp_(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr)
                                
                                #Delta_pl, mat_params                   = explicit_integration(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr)
                                
                                
                                #Delta_pl_sigma33                       = mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement                                                                                   
                                #sigma_i_1                              = sigma_33_trial - (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*(stress_dev_trial[2][2]-stress_back_trial[2][2]))/stress_eqv_trial)
                                
                                #sigma_i_1                              = sigma_33_trial - (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_sigma33[2][2]                                                                                   #amp                                                           = np.array([1.5, 1.5, 1.5, 3, 3, 3])
                                #sigma_i_1                                                     = sigma_33_trial - mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2:3,:].dot(amp*tens_to_voigt(Delta_pl*N_dirr))[0]
                                
                                #sigma_i                                = sigma_i_1
                                #delta_eps33_i_minus_1                  = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] 
                                #mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = -(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] + mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])
                                #mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]  = 0.5*mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1]
                                #delta_eps33_i                          = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]

                        
                        
#...............................................................Updating Variables....................................................................
                        
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        
                        a1                                                            = ML.tens_to_voigt(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        b1                                                            = ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        
#                         N_dirr[0][1]= N_dirr_bef[0][1]/stress_eqv_trial
#                         N_dirr[0][2]= N_dirr_bef[0][2]/stress_eqv_trial
#                         N_dirr[1][0]= N_dirr_bef[1][0]/stress_eqv_trial
#                         N_dirr[1][2]= N_dirr_bef[1][2]/stress_eqv_trial
#                         N_dirr[2][0]= N_dirr_bef[2][0]/stress_eqv_trial
#                         N_dirr[2][1]= N_dirr_bef[2][1]/stress_eqv_trial
                        
                        
                        #N_dirr                                                        = (stress_dev_trial-stress_back_trial)/(stress_eqv_trial)
                        #c1                                                            = ML.tens_to_voigt(N_dirr)
                        
                        #mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*Delta_pl
                        
                        #print(N_dirr)                                                                                                                       
                        
                        
                        
                        
                        #Delta_D                                                       = (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  + (3-6*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*(sph_/J_s)**(2))*(Delta_pl)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
                        #Delta_X                                                       = ((2/3)*mat_params['a'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*voigt_to_tens(c1*1.5) - mat_params['s'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl)                                                                                                                                                                                             
                        
                        
                        #stress_to_be_removed                                          = mat_params['mu1111']*np.array([3*(Delta_pl)*(c1[0]/1), 3*(Delta_pl)*(c1[1]/1), 3*(Delta_pl)*(c1[2]/1), 1.5*(Delta_pl)*(c1[3]/1), 1.5*(Delta_pl)*(c1[4]/1), 1.5*(Delta_pl)*(c1[5]/1)])                                                                                                                       
                                                                                         
                        mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['D_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement)
                        
                        
                        
                        mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['X_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*timeincrement
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - tens_to_voigt(mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*timeincrement) + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       +     (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*(mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - tens_to_voigt(1.5*Delta_pl*N_dirr)))
                        
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  +  mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*c1
                                     
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = ML.tens_to_voigt(stress_dev_trial + stress_sph) - stress_to_be_removed
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #print('Wir befinden uns im Nicht-Linearen bereich des Zeitinkrements: '+str(t+1))
                        mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      = mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*timeincrement 
                        
                        
                        stress_dev_up, stress_sph_                                  = split(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        
                        stress_eqv_up                                                 = np.sqrt(1.5*(np.tensordot(stress_dev_up-mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], stress_dev_up-mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))    
                        
                        stress_sph_up                                                 = (1/3)*(stress_sph_[0][0] + stress_sph_[1][1] + stress_sph_[2][2])
                        
                        mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]     = (((stress_eqv_up/(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N)
                        
                        N_dirr                                                        = (stress_dev_up-mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/stress_eqv_up
                        
                        mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  = 1.5*(mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*N_dirr
                        
                        mat_params['X_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       = ((2/3)*mat_params['a'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['eps_pl_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['s'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        
                        mat_params['D_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       = (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  + (3-6*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*(stress_sph_up/stress_eqv_up)**(2))*(mat_params['eps_dot'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #mat_params['stress_n1111'][2]                                     = 0
                        #mat_params['stress_back1111'][2][2]                               = 0  
                        
                        
                    else:
                        
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  + (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1)).dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot((mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] 
                        mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                        mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        eps33_ana.append(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2])
                        #print(mat_params['stress_n1111'])
                        #print('Wir befinden uns im Linearen bereich des Zeitinkrements: '+str(t+1))
                        #count_tr = 0
                                                                                                                                                          
                                                                                                                                               

        
        

......................................................................................................
Load Sequence 1 The sign is 1
......................................................................................................
......................................................................................................
Load Sequence 1 The sign is 1
......................................................................................................
......................................................................................................
Load Sequence 1 The sign is 1
......................................................................................................


In [ ]:
mat_params['C1111'][2][0]*mat_params['delta_eps1111'][0] + mat_params['C1111'][2][1]*mat_params['delta_eps1111'][1] + mat_params['C1111'][2][2]*mat_params['delta_eps1111'][2]

In [ ]:
mat_params['delta_eps1111']

In [ ]:
mat_params['C1111'].dot(mat_params['delta_eps1111'])

In [ ]:
mat_params['C1111'][2][0]*mat_params['delta_eps1111'][0] + mat_params['C1111'][2][1]*mat_params['delta_eps1111'][1] + mat_params['C1111'][2][2]*mat_params['delta_eps1111'][2]

In [ ]:
mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps1111'])

In [12]:
for i in range(6):
    
    np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Data\test_daten_2nd_spannung_NN'+str(i+1)+'.csv', np.array(mat_params['abs_stress1111'])[:,i:i+1], delimiter=',')
    np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Data\test_daten_2nd_dehnung_NN'+str(i+1)+'.csv', np.array(mat_params['abs_strain1111'])[:,i:i+1], delimiter=',')

In [13]:
for i in range(6):
    
    mat_params['strain'+str(i+1)] = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Data\test_daten_2nd_dehnung'+str(i+1)+'.csv', delimiter=',')
    mat_params['stress'+str(i+1)] = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Data\test_daten_2nd_spannung'+str(i+1)+'.csv', delimiter=',')
    

In [14]:
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111)

ax.plot(np.array(mat_params['abs_strain1111'])[:,0:1], np.array(mat_params['abs_stress1111'])[:,0:1], label='$\sigma_{11}$ NN solution')
ax.plot(np.array(mat_params['abs_strain1111'])[:,1:2], np.array(mat_params['abs_stress1111'])[:,1:2], label='$\sigma_{22}$ NN solution')
#ax.plot(np.array(mat_params['abs_strain1111'])[:,2:3], np.array(mat_params['abs_stress1111'])[:,2:3], label='$\sigma_{33}$ mit Schaedigung_NN')
ax.plot(np.array(mat_params['abs_strain1111'])[:,3:4], np.array(mat_params['abs_stress1111'])[:,3:4], label='$\sigma_{23}$ NN solution')
#ax.plot(np.array(mat_params['abs_strain1111'])[:,4:5], np.array(mat_params['abs_stress1111'])[:,4:5], label='$\sigma_{13}$ mit Schaedigung_NN')
#ax.plot(np.array(mat_params['abs_strain1111'])[:,5:6], np.array(mat_params['abs_stress1111'])[:,5:6], label='$\sigma_{12}$ mit Schaedigung_NN')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,3:4], s23, label='Stress4 mit Schaedigung')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,4:5], s13, label='Stress5 mit Schaedigung')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,5:6], s12, label='Stress6 mit Schaedigung')
plt.grid(b=True, which='major', color='gray', alpha=0.6, linestyle='dashdot', lw=1.5)
    #minor grid lines
plt.minorticks_on()
ax.plot(np.array(mat_params['strain1'])[:], np.array(mat_params['stress1'])[:], label='\sigma_{11}$ solution')
ax.plot(np.array(mat_params['strain2'])[:], np.array(mat_params['stress2'])[:], label='\sigma_{22}$ solution')
#ax.plot(np.array(mat_params['strain3'])[:], np.array(mat_params['stress3'])[:], label='Stress_mit_schaedigung_3')
ax.plot(np.array(mat_params['strain4'])[:], np.array(mat_params['stress4'])[:], label='$\sigma_{23}$ solution')
#ax.plot(np.array(mat_params['strain5'])[:], np.array(mat_params['stress5'])[:], label='$\sigma_{13}$ mit_schaedigung')
#ax.plot(np.array(mat_params['strain6'])[:], np.array(mat_params['stress6'])[:], label='$\sigma_{13}$ mit_schaedigung')
#ax.scatter(np.linspace(1, len(deqpl1), len(deqpl1)), deqpl1)
#ax.scatter(np.linspace(1, len(deqpl2), len(deqpl2)), deqpl2)
# ax.scatter(np.linspace(1, len(eps33_ana), len(eps33_ana)), eps33_ana, label='Elastic')
# ax.scatter(np.linspace(len(eps33_ana), len(eps33_ana_pl)+len(eps33_ana), len(eps33_ana_pl)), eps33_ana_pl, label='Plastic')
# ax.scatter(np.linspace(1, el_.shape[0], el_.shape[0]), el_, label='Elastic_5e-4')
# ax.scatter(np.linspace(el_.shape[0], pl_.shape[0]+el_.shape[0], pl_.shape[0]), pl_, label='Elastic_5e-4')

plt.xlabel('Dehnung')
plt.ylabel('Spannung')
plt.legend()
plt.show()


In [ ]:
mat_params['D1111']

In [ ]:
mat_params['EQPLAS1111']

In [ ]:
np.array(mat_params['abs_strain1111']).shape

In [ ]:
delta_st

In [ ]:
N_dirr[0][2]

In [ ]:
mat_params['stress_back1111']

In [ ]:
mat_params['stress_eqv_trial1111']

In [ ]:
mat_params['C1111'].dot(1.5*Delta_pl*c1)

In [ ]:
t

In [ ]:
stress_dev_trial - 3*mat_params['mu1111']*Delta_pl*N_dirr + stress_sph

In [ ]:
mat_params['delta_eps1111']

In [ ]:
stress_sph

In [ ]:
stress_dev_trial

In [ ]:
eps_dev, eps_sph= split(mat_params['delta_eps1111'])

In [ ]:
tens_to_voigt(eps_dev*2*mat_params['mu1111'] + eps_sph*K) + mat_params['stress_n-11111']

In [ ]:
eps_sph*K

In [ ]:
stress_sph

In [ ]:
(K - (2/3)*mat_params['mu1111'])*(mat_params['delta_eps1111'][0] + mat_params['delta_eps1111'][1]+mat_params['delta_eps1111'][2])

In [ ]:
stress_sph

In [ ]:
sig_tr_i_1

In [ ]:
x_batch_train

In [ ]:
75*13*5

In [ ]:
11.75 - 6.81 + 1.5

In [ ]:
4.44 +2.25+0.75

In [ ]:
9.5+7.44

In [ ]:
5*0.31 + 5*0.90

In [ ]:
25

In [ ]:
((550+500)*90)

In [17]:
components = [11,22,33,23,13,12]

In [19]:
for j in range(6):

    fig = plt.figure(figsize=(6,5))
    #ax = fig.add_subplot(111)
    #plt.plot(mat_params['abs_strain1311'],mat_params['abs_stress1311'],label='Fifth Gauss Point')
    #plt.scatter(np.array(fe_sol_strain_ph)[0][:,j:j+1], np.array(fe_sol_stress_ph)[0][:,j:j+1], label='NN-FE solution with hole')
    #plt.scatter(np.array(fe_sol_strain_woh_ph)[0][:,j:j+1], np.array(fe_sol_stress_woh_ph)[0][:,j:j+1], label='NN-FE Solution')
    plt.plot(np.array(mat_params['abs_strain1111'])[:,j:j+1], np.array(mat_params['abs_stress1111'])[:,j:j+1], label='NN-integration scheme', color='tab:blue')
    plt.plot(mat_params['strain'+str(j+1)], mat_params['stress'+str(j+1)], label='Integration scheme', color='tab:orange')
    plt.legend(loc='lower right')
    #major grid lines
    plt.grid(b=True, which='major', color='gray', alpha=0.6, linestyle='dashdot', lw=1.5)
    #minor grid lines
    plt.minorticks_on()
    plt.grid(b=True, which='minor', color='beige', alpha=0.8, ls='-', lw=1)
    plt.xlabel('$\mathbf{\epsilon_{'+str(components[j])+'}}$', fontsize='14.5')
    plt.ylabel('$\mathbf{\sigma_{'+str(components[j])+'}}$ (MPa)', fontsize='14.5')
    plt.ticklabel_format(axis='both', style='sci', scilimits=(0,0))
    plt.xticks(fontsize='10')
    
    plt.yticks(fontsize='10')
    #plt.title('Material Behaviour')
    #plt.show()
    plt.legend(fontsize='13', loc='best')
    plt.savefig(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Graphs\Sigma_Graph2'+str(components[j])+'drstlng.svg', format='svg')
    plt.savefig(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Graphs\Sigma_Graph2'+str(components[j])+'drstlng.eps', format='eps')
    plt.savefig(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Schädigung_MRC\Graphs\Sigma_Graph2'+str(components[j])+'drstlng.pdf', format='pdf')
     

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript back

In [ ]:
np.array(mat_params['abs_stress1111'])[:,0:1] - mat_params['stress1']

In [ ]:
((np.array(mat_params['abs_strain1111'])[:,0:1] -  mat_params['stress1'].reshape(1200000,1)).mean())/1200000

In [ ]:
mat_params['stress1'].shape

In [ ]:
tf.reduce_sum(tf.square(
   mat_params['stress1'].reshape(1200000,1)- np.array(mat_params['abs_strain1111'])[:,0:1]
))/1200000


In [ ]:
np.sqrt(4144.6119207868305/1200000)

In [ ]:
(tf.square(mat_params['stress1'].reshape(1200000,1) - np.array(mat_params['abs_stress1111'])[:,0:1]))

In [ ]:
np.array(mat_params['abs_stress1111'])[:,0:1]

In [28]:
np.sqrt(tf.keras.losses.MSE(np.array(mat_params['abs_stress1111'])[:,5:6], mat_params['stress6'].reshape(1500000,1)).numpy().mean()/1500000)

0.0005654934122656343

In [ ]:
np.sqrt(1.7929671804047848e-12)